In [ ]:
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices("GPU"))

In [ ]:
zip_url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
urllib.request.urlretrieve(zip_url, "jena_climate.zip")
with zipfile.ZipFile("jena_climate.zip") as archive:
    archive.extractall(".")

weather = pd.read_csv("jena_climate_2009_2016.csv")
weather = weather.iloc[5::6].reset_index(drop=True)
weather["Date Time"] = pd.to_datetime(weather["Date Time"], format="%d.%m.%Y %H:%M:%S")
print(weather.shape)
display(weather.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7))
axes[0].plot(weather["Date Time"], weather["T (degC)"], linewidth=0.4)
axes[0].set_title("Hourly temperature 2009-2016")
axes[1].plot(weather["Date Time"][:240], weather["T (degC)"][:240])
axes[1].set_title("First 10 days")
plt.tight_layout()
plt.show()
display(weather.describe().T)

In [ ]:
for column in ["wv (m/s)", "max. wv (m/s)"]:
    weather.loc[weather[column] < 0, column] = 0.0

wind_direction = np.deg2rad(weather["wd (deg)"])
weather["wind_x"] = weather["wv (m/s)"] * np.cos(wind_direction)
weather["wind_y"] = weather["wv (m/s)"] * np.sin(wind_direction)

timestamp_seconds = weather["Date Time"].astype("int64") // 10**9
day_seconds = 24 * 60 * 60
year_seconds = 365.2425 * day_seconds
weather["day_sin"] = np.sin(timestamp_seconds * (2 * np.pi / day_seconds))
weather["day_cos"] = np.cos(timestamp_seconds * (2 * np.pi / day_seconds))
weather["year_sin"] = np.sin(timestamp_seconds * (2 * np.pi / year_seconds))
weather["year_cos"] = np.cos(timestamp_seconds * (2 * np.pi / year_seconds))

plt.figure(figsize=(12, 3))
plt.plot(weather["day_sin"][:72], label="day sin")
plt.plot(weather["day_cos"][:72], label="day cos")
plt.title("Time of day encoding")
plt.legend()
plt.show()

In [ ]:
feature_columns = ["p (mbar)", "T (degC)", "rh (%)", "VPdef (mbar)", "sh (g/kg)", "rho (g/m**3)", "wind_x", "wind_y", "day_sin", "day_cos", "year_sin", "year_cos"]
feature_values = weather[feature_columns].values.astype("float32")
temperature = weather["T (degC)"].values.astype("float32")

total_rows = len(feature_values)
train_end = int(total_rows * 0.7)
val_end = int(total_rows * 0.9)

feature_mean = feature_values[:train_end].mean(axis=0)
feature_std = feature_values[:train_end].std(axis=0)
normalized_features = (feature_values - feature_mean) / feature_std
temperature_index = feature_columns.index("T (degC)")
print("train rows:", train_end, "validation rows:", val_end - train_end, "test rows:", total_rows - val_end)

In [ ]:
sequence_length = 120
forecast_horizon = 24
batch_size = 256
delay = sequence_length + forecast_horizon - 1

def make_dataset(start_index, end_index, shuffle):
    return keras.utils.timeseries_dataset_from_array(
        normalized_features[:-delay],
        targets=temperature[delay:].reshape(-1, 1),
        sequence_length=sequence_length,
        shuffle=shuffle,
        batch_size=batch_size,
        start_index=start_index,
        end_index=end_index
    )

train_dataset = make_dataset(0, train_end, True)
val_dataset = make_dataset(train_end, val_end, False)
test_dataset = make_dataset(val_end, None, False)

for sample_windows, sample_targets in train_dataset.take(1):
    print("input windows:", sample_windows.shape, "targets:", sample_targets.shape)

In [ ]:
def naive_forecast_mae(dataset):
    total_error, total_count = 0.0, 0
    for windows, targets in dataset:
        last_temperature = windows[:, -1, temperature_index] * feature_std[temperature_index] + feature_mean[temperature_index]
        total_error += float(tf.reduce_sum(tf.abs(last_temperature - tf.reshape(targets, [-1]))))
        total_count += int(windows.shape[0])
    return total_error / total_count

naive_val_mae = naive_forecast_mae(val_dataset)
naive_test_mae = naive_forecast_mae(test_dataset)
print(f"naive baseline (temperature stays the same) validation MAE {naive_val_mae:.3f}  test MAE {naive_test_mae:.3f}")

In [ ]:
feature_count = len(feature_columns)

def build_dense_model():
    return keras.Sequential([
        layers.Input(shape=(sequence_length, feature_count)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(1)
    ], name="dense")

def build_lstm_model():
    return keras.Sequential([
        layers.Input(shape=(sequence_length, feature_count)),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(1)
    ], name="lstm")

def build_stacked_lstm_model():
    return keras.Sequential([
        layers.Input(shape=(sequence_length, feature_count)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(32),
        layers.Dropout(0.3),
        layers.Dense(1)
    ], name="stacked_lstm")

def build_cnn_model():
    return keras.Sequential([
        layers.Input(shape=(sequence_length, feature_count)),
        layers.Conv1D(32, 24, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 12, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 6, activation="relu"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(1)
    ], name="cnn_1d")

def build_cnn_lstm_model():
    return keras.Sequential([
        layers.Input(shape=(sequence_length, feature_count)),
        layers.Conv1D(64, 5, padding="causal", activation="relu"),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, 3, padding="causal", activation="relu"),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(1)
    ], name="cnn_lstm")

model_builders = {
    "Dense": build_dense_model,
    "LSTM": build_lstm_model,
    "Stacked LSTM": build_stacked_lstm_model,
    "1D CNN": build_cnn_model,
    "CNN + LSTM": build_cnn_lstm_model
}
build_cnn_lstm_model().summary()

In [ ]:
forecast_histories = {}
forecast_models = {}
for name, builder in model_builders.items():
    keras.utils.set_random_seed(42)
    model = builder()
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=20,
        verbose=0,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_mae", patience=4, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
        ]
    )
    forecast_histories[name] = history.history
    forecast_models[name] = model
    print(f"{name:<13} epochs {len(history.history['loss']):>2}  best val MAE {min(history.history['val_mae']):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for name, history in forecast_histories.items():
    axes[0].plot(history["mae"], label=name)
    axes[1].plot(history["val_mae"], label=name)
for axis, title in zip(axes, ["Training MAE", "Validation MAE"]):
    axis.axhline(naive_val_mae, color="black", linestyle="--", label="naive baseline")
    axis.set_title(title)
    axis.set_xlabel("epoch")
    axis.set_ylabel("degrees C")
    axis.legend()
plt.show()

In [ ]:
comparison_rows = [{"model": "naive baseline", "test MAE": naive_test_mae, "parameters": 0}]
for name, model in forecast_models.items():
    test_mse, test_mae = model.evaluate(test_dataset, verbose=0)
    comparison_rows.append({"model": name, "test MAE": test_mae, "parameters": model.count_params()})
comparison_frame = pd.DataFrame(comparison_rows).set_index("model").round(3)
display(comparison_frame)

comparison_frame["test MAE"].plot(kind="bar", figsize=(10, 4.5), rot=0, color="slateblue")
plt.ylabel("degrees C")
plt.title(f"Test MAE for {forecast_horizon} hour ahead temperature forecast")
plt.show()

In [ ]:
true_test_temperature = np.concatenate([targets.numpy().flatten() for _, targets in test_dataset])
shown_hours = 500

plt.figure(figsize=(16, 5))
plt.plot(true_test_temperature[:shown_hours], label="actual", color="black", linewidth=2)
for name in ["LSTM", "1D CNN", "CNN + LSTM"]:
    predictions = forecast_models[name].predict(test_dataset, verbose=0).flatten()
    plt.plot(predictions[:shown_hours], label=name, alpha=0.8)
plt.title(f"Test set: actual vs predicted temperature {forecast_horizon} hours ahead")
plt.xlabel("hour")
plt.ylabel("degrees C")
plt.legend()
plt.show()

In [ ]:
best_name = comparison_frame.drop("naive baseline")["test MAE"].idxmin()
best_predictions = forecast_models[best_name].predict(test_dataset, verbose=0).flatten()
forecast_errors = best_predictions - true_test_temperature

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].hist(forecast_errors, bins=60, color="teal")
axes[0].set_title(f"{best_name} forecast error distribution")
axes[0].set_xlabel("predicted minus actual (degrees C)")
axes[1].scatter(true_test_temperature, best_predictions, s=2, alpha=0.3)
axes[1].plot([-20, 35], [-20, 35], color="red")
axes[1].set_xlabel("actual")
axes[1].set_ylabel("predicted")
axes[1].set_title(f"{best_name} predicted vs actual")
plt.show()

forecast_models[best_name].save("best_temperature_forecaster.keras")
print("best model:", best_name)